In [ ]:
from google.colab import drive
drive.mount("/content/gdrive", force_remount=True)

Mounted at /content/gdrive


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>02 — تحديد مسارات النماذج وملفات الواجهة</h2>

  <p>
    تحدد هذه الخلية مسارات النموذج النهائي
    <span dir="ltr">Final Fine-Tuned MiniLM Comment Encoder</span>
    ونموذج
    <span dir="ltr">BERTopic</span>
    وملف
    <span dir="ltr">topic_info</span>.
  </p>

  <p>
    الهدف هو التأكد من أن جميع الملفات اللازمة لتشغيل واجهة
    <span dir="ltr">Streamlit</span>
    موجودة قبل إنشاء التطبيق وتشغيله.
  </p>
</div>

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/gdrive/MyDrive/semester project/news_comment_topic_system")

for p in PROJECT_ROOT.rglob("*.npy"):
    if "v7" in p.name.lower() or "embedding" in p.name.lower():
        print(p)

/content/gdrive/MyDrive/semester project/news_comment_topic_system/bertopic_committee_demo_outputs/v5_stage1_test_large_embeddings.npy
/content/gdrive/MyDrive/semester project/news_comment_topic_system/bertopic_v6b_seq256_evaluation_outputs/v6b_seq256_test_large_embeddings.npy
/content/gdrive/MyDrive/semester project/news_comment_topic_system/bertopic_v6b_seq256_evaluation_outputs/test_large_embeddings.npy
/content/gdrive/MyDrive/semester project/news_comment_topic_system/outputs/benchmark_50k_fixed_20260420_190627/baseline_embeddings.npy
/content/gdrive/MyDrive/semester project/news_comment_topic_system/outputs/benchmark_50k_fixed_20260420_190627/v3_embeddings.npy
/content/gdrive/MyDrive/semester project/news_comment_topic_system/outputs/benchmark_50k_fixed_20260420_190627/v4_embeddings.npy
/content/gdrive/MyDrive/semester project/news_comment_topic_system/models/v8_baseline_minilm_largepairs_epochs10_es3_lr1e5/bertopic_v8_umap15_mcs100_ms10/embeddings_v8_test_large.npy
/content/gdriv

<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>03 — تثبيت مكتبات الواجهة</h2>

  <p>
    تثبّت هذه الخلية المكتبات اللازمة لتشغيل واجهة
    <span dir="ltr">Streamlit</span>
    وتحميل نموذج
    <span dir="ltr">SentenceTransformer</span>
    ونموذج
    <span dir="ltr">BERTopic</span>
    المحفوظ.
  </p>

  <p>
    هذه الخلية خاصة بالواجهة فقط، ولا تعيد تدريب النموذج ولا تعيد استخراج المواضيع من جديد.
  </p>
</div>

In [ ]:
!pip install -q streamlit bertopic sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 114.7 MB/s eta 0:00:00


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>04 — إنشاء ملف واجهة Streamlit</h2>

  <p>
    تنشئ هذه الخلية ملف
    <span dir="ltr">app.py</span>
    الذي يحتوي على واجهة بسيطة لإدخال تعليق إخباري وتحليله باستخدام النموذج النهائي.
  </p>

  <p>
    الواجهة تحمّل
    <span dir="ltr">Final Fine-Tuned MiniLM Comment Encoder</span>
    ونموذج
    <span dir="ltr">BERTopic</span>
    المحفوظ، ثم تربط التعليق الجديد بأقرب موضوع مكتشف مسبقًا.
  </p>

  <p>
    هذه الخلية لا تعيد التدريب ولا تعيد بناء
    <span dir="ltr">BERTopic</span>،
    بل تستخدم النماذج المحفوظة فقط في مرحلة
    <span dir="ltr">inference</span>.
  </p>
</div>

In [ ]:
%%writefile app.py
import re
from pathlib import Path

import numpy as np
import pandas as pd
import streamlit as st

from sentence_transformers import SentenceTransformer
from bertopic import BERTopic


PROJECT_ROOT = Path("/content/gdrive/MyDrive/semester project/news_comment_topic_system")

if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path("/content/drive/MyDrive/semester project/news_comment_topic_system")

FINAL_EMBEDDING_MODEL = PROJECT_ROOT / "models/final_finetuned_minilm_comment_encoder"

FINAL_EVAL_DIR = (
    PROJECT_ROOT
    / "final_bertopic_evaluation_outputs"
    / "final_minilm_comment_encoder_umap15_mcs100_ms10"
)

FINAL_BERTOPIC_MODEL = FINAL_EVAL_DIR / "final_bertopic_model"
FINAL_TOPIC_INFO = FINAL_EVAL_DIR / "final_topic_info.csv"


def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " url ", text)
    text = re.sub(r"\S+@\S+", " email ", text)
    text = re.sub(r"\d+", " number ", text)
    text = re.sub(r"[^a-zA-Z\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


@st.cache_resource
def load_embedding_model():
    model = SentenceTransformer(str(FINAL_EMBEDDING_MODEL))
    model.max_seq_length = 256
    return model


@st.cache_resource
def load_topic_model():
    return BERTopic.load(str(FINAL_BERTOPIC_MODEL))


@st.cache_data
def load_topic_info():
    if FINAL_TOPIC_INFO.exists():
        return pd.read_csv(FINAL_TOPIC_INFO)
    return None


def get_confidence(probs):
    if probs is None:
        return None

    arr = np.asarray(probs)

    if arr.ndim == 1:
        return float(arr[0])

    if arr.ndim == 2:
        return float(np.max(arr[0]))

    return None


def get_topic_name(topic_info, topic_id):
    if topic_id == -1:
        return "Noise / Unclear Topic"

    if topic_info is None:
        return f"Topic {topic_id}"

    row = topic_info[topic_info["Topic"] == topic_id]

    if row.empty:
        return f"Topic {topic_id}"

    if "Name" in row.columns:
        return str(row.iloc[0]["Name"])

    return f"Topic {topic_id}"


def get_topic_keywords(topic_model, topic_id):
    if topic_id == -1:
        return []

    words = topic_model.get_topic(topic_id)

    if not words:
        return []

    return [word for word, score in words[:10]]


st.set_page_config(
    page_title="News Comment Topic Analysis",
    page_icon="🧠",
    layout="centered"
)

st.title("News Comment Topic Analysis System")
st.caption("Final Fine-Tuned MiniLM Comment Encoder + BERTopic")

st.write(
    "اكتب تعليقًا إخباريًا، وسيقوم النظام بربطه بأقرب موضوع مكتشف سابقًا."
)

comment = st.text_area(
    "Comment",
    height=160,
    placeholder="Example: The healthcare system is broken and insurance companies are making things worse."
)

if st.button("Analyze", type="primary"):
    if not comment.strip():
        st.warning("يرجى كتابة تعليق أولًا.")
    else:
        embedding_model = load_embedding_model()
        topic_model = load_topic_model()
        topic_info = load_topic_info()

        cleaned = clean_text(comment)

        embedding = embedding_model.encode(
            [cleaned],
            normalize_embeddings=True
        )

        predicted_topics, probs = topic_model.transform(
            [cleaned],
            embeddings=embedding
        )

        topic_id = int(predicted_topics[0])
        confidence = get_confidence(probs)

        st.subheader("Result")

        col1, col2 = st.columns(2)

        with col1:
            st.metric("Topic ID", topic_id)

        with col2:
            if confidence is None:
                st.metric("Confidence", "N/A")
            else:
                st.metric("Confidence", f"{confidence * 100:.2f}%")

        if topic_id == -1:
            st.warning("لم يتم ربط التعليق بموضوع واضح. النتيجة Noise / Outlier.")
        else:
            topic_name = get_topic_name(topic_info, topic_id)
            keywords = get_topic_keywords(topic_model, topic_id)

            st.write("Topic name:")
            st.code(topic_name)

            st.write("Top keywords:")
            st.write(", ".join(keywords))

        with st.expander("Cleaned text"):
            st.write(cleaned)

Writing app.py


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>05 — تشغيل واجهة Streamlit في الخلفية</h2>

  <p>
    تشغّل هذه الخلية ملف
    <span dir="ltr">app.py</span>
    على المنفذ
    <span dir="ltr">8502</span>
    باستخدام
    <span dir="ltr">Streamlit</span>.
  </p>

  <p>
    يتم التشغيل في الخلفية باستخدام
    <span dir="ltr">nohup</span>
    حتى

In [ ]:
!nohup streamlit run app.py \
  --server.port 8502 \
  --server.address 0.0.0.0 \
  --server.headless true \
  --server.enableCORS false \
  --server.enableXsrfProtection false \
  > streamlit.log 2>&1 &

<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>06 — فحص تشغيل واجهة Streamlit</h2>

  <p>
    تتحقق هذه الخلية من أن واجهة
    <span dir="ltr">Streamlit</span>
    تعمل على المنفذ
    <span dir="ltr">8502</span>.
  </p>

  <p>
    تعرض آخر رسائل التشغيل من ملف
    <span dir="ltr">streamlit.log</span>
    ثم تختبر الاتصال المحلي بالواجهة باستخدام
    <span dir="ltr">curl</span>.
  </p>
</div>

In [ ]:
!sleep 5
!tail -n 20 streamlit.log
!curl -I http://127.0.0.1:8502



2026-05-04 03:01:03.031 Uvicorn server started on 0.0.0.0:8502

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8502
  Network URL: http://172.28.0.12:8502
  External URL: http://34.124.189.159:8502

HTTP/1.1 200 OK
date: Mon, 04 May 2026 03:01:17 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 5381
last-modified: Mon, 04 May 2026 02:50:28 GMT
etag: "2dbf04bc414f2a4e18ba3a11bf747c37"
cache-control: no-cache



<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>07 — إنشاء رابط خارجي للواجهة</h2>

  <p>
    تحمّل هذه الخلية أداة
    <span dir="ltr">cloudflared</span>
    ثم تنشئ
    <span dir="ltr">Cloudflare Tunnel</span>
    يربط المنفذ المحلي
    <span dir="ltr">8502</span>
    برابط خارجي يمكن فتحه من المتصفح.
  </p>

  <p>
    يجب تشغيل واجهة
    <span dir="ltr">Streamlit</span>
    وفحصها محليًا قبل تشغيل هذه الخلية.
  </p>
</div>

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared tunnel --url http://127.0.0.1:8502

2026-05-04T03:02:02Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-05-04T03:02:02Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-05-04T03:02:07Z INF +--------------------------------------------------------------------------------------------+
2026-05-04T03:02:07Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-05-04T03:02:07Z INF |  https://columns-bureau-hope-threaded.trycloudflare.co